# 01 Baseline Replay
Thin notebook: orchestration only. Business logic lives in src/.

In [ ]:
import os
import sys
import importlib
from IPython.display import display

cwd = os.getcwd()
candidates = [cwd, os.path.dirname(cwd)]
REPO = next((p for p in candidates if os.path.exists(os.path.join(p, 'src'))), cwd)
SRC = os.path.join(REPO, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import margin_analysis
importlib.reload(margin_analysis)
import charts
importlib.reload(charts)
import replay_analyzer
importlib.reload(replay_analyzer)
from replay_analyzer import default_analyzer_for_repo

In [ ]:
analyzer = default_analyzer_for_repo(REPO)
baseline = analyzer.replay_folder(os.path.join(REPO, 'data', 'baseline'), export_dir=os.path.join(REPO, 'output'))

print('Detected Baseline Configuration')
display(baseline.configuration())

print('Summary Metrics')
display(baseline.summary())

print('Per Pair Statistics')
display(baseline.pair_table())

In [ ]:
import pandas as pd
import margin_analysis
import charts

ma = baseline.margin_analysis()
events = ma.events_df()
threshold_summary = ma.threshold_summary_df(events)

print('Portfolio Margin Summary')
display(pd.DataFrame([ma.portfolio_summary_dict(events)]).T.rename(columns={0: 'value'}))

print('\nThreshold Summary (Canonical Events)')
display(threshold_summary)

print('\nPer-Pair Contribution Summary')
display(ma.pair_summary_df(events))

if not events.empty:
    print(f'\nCanonical Margin Events ({len(events)} episodes) — worst 10 by margin level')
    display(events.nsmallest(10, 'min_margin_level')[
        ['event_id', 'start_time', 'end_time', 'duration_minutes',
         'min_margin_level', 'worst_threshold_crossed', 'thresholds_crossed',
         'max_used_margin', 'largest_used_margin_contributor',
         'largest_used_margin_contribution_pct']
    ])

    print('\nBasket Composition at 3 Worst Events')
    display(ma.event_baskets_df(events.nsmallest(3, 'min_margin_level')))

## Margin Analysis

In [ ]:
fig_equity = baseline.plot_equity()
fig_pair_balance = baseline.plot_pair_contributions()
fig_pair_floating = baseline.plot_pair_floating()
fig_pair_drawdown = baseline.plot_pair_drawdown()
fig_margin = baseline.plot_margin()

fig_equity.show()
fig_pair_balance.show()
fig_pair_floating.show()
fig_pair_drawdown.show()
fig_margin.show()

## Swap Reports
Focused swap diagnostics for the baseline replay.

In [ ]:
import os
import pandas as pd
from IPython.display import display

# Clean up stale swap_position_summary.csv from previous runs with different pairs
pos_path = os.path.join(REPO, 'output', 'swap_position_summary.csv')
if os.path.exists(pos_path):
    os.remove(pos_path)
    print(f'Removed stale {pos_path} to prevent displaying outdated position data.')

replay_df = baseline.replay_table().copy()
replay_df['time_dt'] = pd.to_datetime(replay_df['time'], format='%Y.%m.%d %H:%M:%S')
swap_df = replay_df[replay_df['EventType'] == 'swap'].copy()

if swap_df.empty:
    print('No swap rows were found in the current baseline replay output.')
else:
    swap_df['post_date'] = swap_df['time_dt'].dt.date
    swap_df['month'] = swap_df['time_dt'].dt.to_period('M').astype(str)

    total_swap = float(swap_df['ModeledSwap'].sum())
    swap_events = int(len(swap_df))
    avg_swap_per_event = float(total_swap / swap_events)
    avg_daily_swap = float(swap_df.groupby('post_date')['ModeledSwap'].sum().mean())

    deal_df = replay_df[replay_df['EventType'].astype(str).str.startswith('deal_')].copy()
    realized_profit = float(deal_df['scaled_net_profit'].sum())
    swap_pct_realized = (total_swap / realized_profit * 100.0) if realized_profit != 0 else float('nan')

    print('Swap summary')
    display(pd.DataFrame([
        {'Metric': 'Total modeled swap', 'Value': total_swap},
        {'Metric': 'Swap events', 'Value': swap_events},
        {'Metric': 'Average swap per event', 'Value': avg_swap_per_event},
        {'Metric': 'Average daily swap', 'Value': avg_daily_swap},
        {'Metric': 'Swap as % of realized profit', 'Value': swap_pct_realized},
    ]))

    print('Swap by pair')
    display(swap_df.groupby('pair', as_index=False)['ModeledSwap'].sum().sort_values('ModeledSwap'))

    print('Swap by direction')
    display(swap_df.groupby('direction', as_index=False)['ModeledSwap'].sum().sort_values('ModeledSwap'))

    print('Monthly swap')
    display(swap_df.groupby('month', as_index=False)['ModeledSwap'].sum().sort_values('month'))

    print('Top 20 positions by total swap')
    if os.path.exists(pos_path):
        pos_df = pd.read_csv(pos_path)
        display(pos_df.sort_values('Total Swap').head(20))
    else:
        print('swap_position_summary.csv not found in output/. Run swap_audit_report.py to generate it.')

    print('Cumulative swap over time (last 20 days)')
    daily = swap_df.groupby('post_date', as_index=False)['ModeledSwap'].sum().sort_values('post_date')
    daily['CumulativeSwap'] = daily['ModeledSwap'].cumsum()
    display(daily.tail(20))
